In [1]:
import pandas as pd

In [2]:
tb = pd.read_csv('total_ensemble_table_v2.csv')
print(f"Total edges: {len(tb)}, Agreement score range: {tb.agreement_score.min()}–{tb.agreement_score.max()}")
tb.head()

Total edges: 794, Agreement score range: 1–11


,edge,cause,effect,FCI_fisherz_simple,FCI_fisherz_simple_indicator,FCI_kci_knn,FCI_kci_simple,FCI_mv_fisherz_raw,PC_fisherz_knn_indicator,PC_fisherz_simple,PC_fisherz_simple_indicator,PC_kci_knn_indicator,PC_kci_simple,PC_mv_fisherz_raw,avg_num_paths,agreement_score
0,AKI Onset (24h) --> AKI Post-24h,AKI Onset (24h),AKI Post-24h,1,1,1,1,1,1,1,1,1,1,1,74.1,11
1,Gender --> Platelet (max),Gender,Platelet (max),1,1,1,1,1,1,1,1,1,1,1,39.2,11
2,Mech. Vent Post-24h --> Hospital Mortality,Mech. Vent Post-24h,Hospital Mortality,1,1,1,1,1,1,1,1,1,1,1,0.4,11
3,Antibiotics --> Hospital Mortality,Antibiotics,Hospital Mortality,1,1,1,1,1,1,1,1,1,1,1,684.4,11
4,Mech. Vent Onset (24h) --> Mech. Vent Post-24h,Mech. Vent Onset (24h),Mech. Vent Post-24h,1,1,1,1,1,1,1,1,1,1,1,64.7,11


In [3]:
tb.loc[(tb["edge"]=="AKI Onset (24h) --> Mech. Vent Post-24h") | (tb["edge"]=="Mech. Vent Onset (24h) --> AKI Post-24h"),["edge","agreement_score"]]

,edge,agreement_score
26,AKI Onset (24h) --> Mech. Vent Post-24h,9
28,Mech. Vent Onset (24h) --> AKI Post-24h,8


In [4]:
row = tb.loc[tb["edge"] == "Mech. Vent Onset (24h) --> AKI Post-24h"]
row.T

,28
edge,Mech. Vent Onset (24h) --> AKI Post-24h
cause,Mech. Vent Onset (24h)
effect,AKI Post-24h
FCI_fisherz_simple,1
FCI_fisherz_simple_indicator,0
FCI_kci_knn,1
FCI_kci_simple,0
FCI_mv_fisherz_raw,0
PC_fisherz_knn_indicator,1
PC_fisherz_simple,1


In [5]:
results = []
targets = ["CKD (Baseline)", "CVP (max)", "Fluid Balance"]
for var in targets:
    mask = (tb["cause"] == var) | (tb["effect"] == var) 
    results.append(tb[mask][["edge","agreement_score"]])
for var, res in zip(targets, results):
    print(f"Edges involving {var}:")
    print(res.head(10))
    print()

Edges involving CKD (Baseline):
                                         edge  agreement_score
17     CKD (Baseline) --> Mech. Vent Post-24h               10
20      CKD (Baseline) --> Hospital Mortality               10
23        CKD (Baseline) --> Hemoglobin (min)                9
24        CKD (Baseline) --> Heart Rate (max)                9
32            CKD (Baseline) --> AKI Post-24h                8
39         CKD (Baseline) --> AKI Onset (24h)                7
67         CKD (Baseline) --> Bilirubin (max)                6
81              CKD (Baseline) --> SpO2 (min)                5
83          CKD (Baseline) --> Platelet (max)                5
84  CKD (Baseline) --> Mech. Vent Onset (24h)                5

Edges involving CVP (max):
                                   edge  agreement_score
104    CVP (max) --> Hospital Mortality                5
107            CVP (max) --> FiO2 (max)                5
152      CVP (max) --> Heart Rate (max)                4
155      CVP (max) 

In [6]:
indicator_runs = [c for c in tb.columns 
                  if c not in ["edge","cause","effect","agreement_score","avg_num_paths"]
                  and "indicator" in c]

non_indicator_runs = [c for c in tb.columns 
                      if c not in ["edge","cause","effect","agreement_score","avg_num_paths"]
                      and c not in indicator_runs]

print(f"Indicator runs ({len(indicator_runs)}): {indicator_runs}")
print(f"Non-indicator runs ({len(non_indicator_runs)}): {non_indicator_runs}")

def normalized_score(row):
    is_indicator_edge = any(x in str(row["cause"]) + str(row["effect"]) 
                           for x in ["Missing", "missing"])
    eligible = indicator_runs if is_indicator_edge else indicator_runs + non_indicator_runs
    return row[eligible].sum() / len(eligible)

tb["agreement_norm"] = tb.apply(normalized_score, axis=1)

mask = (tb["edge"] == "AKI Onset (24h) --> Mech. Vent Post-24h") | \
       (tb["edge"] == "Mech. Vent Onset (24h) --> AKI Post-24h")
tb[mask][["edge","agreement_score","agreement_norm"]]

Indicator runs (4): ['FCI_fisherz_simple_indicator', 'PC_fisherz_knn_indicator', 'PC_fisherz_simple_indicator', 'PC_kci_knn_indicator']
Non-indicator runs (7): ['FCI_fisherz_simple', 'FCI_kci_knn', 'FCI_kci_simple', 'FCI_mv_fisherz_raw', 'PC_fisherz_simple', 'PC_kci_simple', 'PC_mv_fisherz_raw']


,edge,agreement_score,agreement_norm
26,AKI Onset (24h) --> Mech. Vent Post-24h,9,0.818182
28,Mech. Vent Onset (24h) --> AKI Post-24h,8,0.727273


In [7]:
consensus = tb[tb["agreement_norm"] > 0.70][["edge","cause","effect","agreement_score","agreement_norm"]].sort_values("agreement_norm", ascending=False)

print(f"{len(consensus)} edges at >0.70 normalized")
print(consensus.to_string())

98 edges at >0.70 normalized
                                               edge                   cause                effect  agreement_score  agreement_norm
0                  AKI Onset (24h) --> AKI Post-24h         AKI Onset (24h)          AKI Post-24h               11        1.000000
156                            Race --> BMI Missing                    Race           BMI Missing                4        1.000000
185             FiO2 Missing --> Hospital Mortality            FiO2 Missing    Hospital Mortality                4        1.000000
186                          Gender --> CVP Missing                  Gender           CVP Missing                4        1.000000
187            FiO2 Missing --> Mech. Vent Post-24h            FiO2 Missing   Mech. Vent Post-24h                4        1.000000
196                          Gender --> INR Missing                  Gender           INR Missing                4        1.000000
200            Lactate Missing --> Hemoglobin (min)   

In [8]:
import networkx as nx
from dowhy import CausalModel

# Build consensus edge list
consensus = tb[tb["agreement_norm"] > 0.70][["cause","effect","agreement_norm"]]

# Build DiGraph
G = nx.DiGraph()
for _, row in consensus.iterrows():
    G.add_edge(row["cause"], row["effect"])

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

# Detect cycles
cycles = list(nx.simple_cycles(G))
print(f"\n{len(cycles)} cycles found:")
for c in cycles:
    print(" --> ".join(c + [c[0]]))

Nodes: 33
Edges: 98

0 cycles found:


In [9]:
consensus.to_csv("consensus_edges.csv", index=False)

In [11]:
for treatment in ["Mech. Vent Onset (24h)", "AKI Onset (24h)"]:
    edges = tb[tb["effect"] == treatment][["edge","agreement_score","agreement_norm"]]\
        .sort_values("agreement_norm", ascending=False)
    print(f"\nAll edges into {treatment}:")
    print(edges.to_string())


All edges into Mech. Vent Onset (24h):
                                          edge  agreement_score  agreement_norm
84   CKD (Baseline) --> Mech. Vent Onset (24h)                5        0.454545
242          Gender --> Mech. Vent Onset (24h)                3        0.272727
591    Temp Missing --> Mech. Vent Onset (24h)                1        0.250000
609     CVP Missing --> Mech. Vent Onset (24h)                1        0.250000
751             Age --> Mech. Vent Onset (24h)                1        0.090909

All edges into AKI Onset (24h):
                                           edge  agreement_score  agreement_norm
39           CKD (Baseline) --> AKI Onset (24h)                7        0.636364
179               INR (max) --> AKI Onset (24h)                4        0.363636
190                  Gender --> AKI Onset (24h)                4        0.363636
171           Lactate (max) --> AKI Onset (24h)                4        0.363636
267             Antibiotics --> AKI Onset 